# Обучение «полной мощью» на Kaggle

Конфигурация: **ResNet50, 512×512, мульти-доменный вход (RGB+шум+DCT), BCE+Dice, TTA**
(86 GFLOPs, лимит 100).

## Перед запуском

1. Прикрепи датасет с данными: справа **Add Input → Datasets → твой датасет**.
2. В Settings включи **Accelerator: GPU T4 x2** (или P100).
3. Репозиторий должен быть публичным (для `git clone`), либо загрузи папку `src/` вручную.
4. Run all.

In [ ]:
import subprocess, sys

def have_cuda_torch():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

# torch ставим только если в окружении нет CUDA-версии
if not have_cuda_torch():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121",
                    "--index-url", "https://download.pytorch.org/whl/cu121"])

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segmentation-models-pytorch==0.4.0", "opencv-python-headless",
                "Pillow", "numpy==1.26.4", "tqdm", "matplotlib", "pandas"])

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
import glob, csv, subprocess, sys
from pathlib import Path

# Автоопределение путей к данным конкурса под /kaggle/input
def find_train_csv():
    m = glob.glob("/kaggle/input/**/stage1/train.csv", recursive=True)
    return m[0] if m else None

def find_test_csv():
    for p in glob.glob("/kaggle/input/**/test.csv", recursive=True):
        try:
            with open(p, encoding="utf-8") as f:
                if "img_path" in f.readline():
                    return p
        except Exception:
            continue
    return None

TRAIN_CSV = find_train_csv()
TEST_CSV = find_test_csv()
assert TRAIN_CSV and TEST_CSV, "Не найдены train.csv/test.csv под /kaggle/input"

TRAIN_DIR = str(Path(TRAIN_CSV).parent.parent)   # .../train_stage1
TEST_DIR = str(Path(TEST_CSV).parent)            # .../test_stage1/test_stage1
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

# Клонируем код решения
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/ssaikk25/cv.git", "/kaggle/working/repo"])
sys.path.insert(0, "/kaggle/working/repo")

# Параметры обучения
params = dict(
    encoder="resnet50",
    img_size=512,
    batch_size=16,
    epochs=15,
    dice_w=1.0,
    neg_ratio=0.2,
    val_neg=1500,
    name="manip_resnet50_512",
    resume=True,
)

## 1. Кэш статистики масок

Отделяет позитивы от чистых строк (негативы для FPR).

In [ ]:
subprocess.run([sys.executable, "-m", "src.prepare",
                "--data-dir", TRAIN_DIR, "--train-csv", TRAIN_CSV,
                "--output", "/kaggle/working/mask_stats.json"],
               cwd="/kaggle/working/repo")

## 2. Обучение

Чекпоинты и подобранный порог сохраняются в `/kaggle/working/checkpoints/`.

In [ ]:
cmd = [sys.executable, "-m", "src.train",
       "--data-dir", TRAIN_DIR,
       "--train-csv", TRAIN_CSV,
       "--mask-stats", "/kaggle/working/mask_stats.json",
       "--img-size", str(params["img_size"]),
       "--batch-size", str(params["batch_size"]),
       "--epochs", str(params["epochs"]),
       "--dice-w", str(params["dice_w"]),
       "--neg-ratio", str(params["neg_ratio"]),
       "--val-neg", str(params["val_neg"]),
       "--encoder", params["encoder"],
       "--num-workers", "2",
       "--checkpoint-dir", "/kaggle/working/checkpoints",
       "--name", params["name"]]
if params.get("resume"):
    cmd += ["--resume"]
subprocess.run(cmd, cwd="/kaggle/working/repo")

## 3. Инференс с TTA и сборка посылки

In [ ]:
cmd = [sys.executable, "-m", "src.predict",
       "--data-dir", TEST_DIR,
       "--test-csv", TEST_CSV,
       "--encoder", params["encoder"],
       "--img-size", str(params["img_size"]),
       "--checkpoint", f"/kaggle/working/checkpoints/{params['name']}_best.pth",
       "--threshold-json", f"/kaggle/working/checkpoints/{params['name']}_threshold.json",
       "--tta",
       "--num-workers", "2",
       "--pred-dir", "/kaggle/working/predictions",
       "--submission-csv", "/kaggle/working/submission.csv",
       "--zip", "/kaggle/working/submission.zip"]
subprocess.run(cmd, cwd="/kaggle/working/repo")

## 4. Готово

Скачай `/kaggle/working/submission.zip` через панель **Output** и загрузи на платформу конкурса.

Если лимит времени сессии близко — уменьши `epochs` (до 8) или `img_size` (до 384).

In [ ]:
import zipfile, os

# Собираем всё важное в один архив, чтобы скачать одной кнопкой
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    if os.path.exists("/kaggle/working/submission.zip"):
        zf.write("/kaggle/working/submission.zip", "submission.zip")
    for f in [f"/kaggle/working/checkpoints/{params['name']}_best.pth",
              f"/kaggle/working/checkpoints/{params['name']}_last.pth",
              f"/kaggle/working/checkpoints/{params['name']}_threshold.json"]:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
print("results.zip готов — скачай через Output. Внутри: submission.zip + чекпоинты")